In [1]:
import ee
import geemap
from dotenv import load_dotenv
import os
load_dotenv(dotenv_path='.env')
project_name = os.getenv("project_name")
# print(f"Project Name: {project_name}")
ee.Initialize(project=project_name)


In [2]:
map = geemap.Map()

point = ee.Geometry.Point([90.4152, 23.8041]) #around dhaka
region = ee.Geometry.Rectangle([90.3, 23.7, 90.5, 23.9]) #bounding box around the point

map.centerObject(point, 10)
map.addLayer(point, {'color': 'red'}, 'Point Layer')
map.addLayer(region, {'color': 'blue'}, 'Region Layer')

map #just testing if everything is working

Map(center=[23.8041, 90.4152], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

In [6]:
#collecting data
s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") #Sentinel-2 Surface Reflectance data

filtered_image = s2 \
    .filterBounds(region) \
    .sort('system:time_start')\
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
    #.filterDate('2022-01-01', '2022-12-31') \
    

print(f"Number of images found: {filtered_image.size().getInfo()}")

Number of images found: 327


In [7]:
#now making sure they have full coverage of the region
strict_collection = filtered_image.filter(ee.Filter.contains(
        leftField='.geo', #.geo refers to the geometry of the image
        rightValue=region #the region we defined earlier 
    )
)

print(f"Number of images with full coverage: {strict_collection.size().getInfo()}")

Number of images with full coverage: 321


In [8]:
min_timestamp = strict_collection.aggregate_min("system:time_start")
max_timestamp = strict_collection.aggregate_max("system:time_start")
print(f"Min Timestamp: {min_timestamp.getInfo()}")
print(f"Max Timestamp: {max_timestamp.getInfo()}")
print('First Date:', ee.Date(min_timestamp).format('YYYY-MM-dd').getInfo())
print('Last Date:', ee.Date(max_timestamp).format('YYYY-MM-dd').getInfo())

Min Timestamp: 1453092113893
Max Timestamp: 1766032934402
First Date: 2016-01-18
Last Date: 2025-12-18


In [9]:

visualized_collection = strict_collection.map(lambda img: img.visualize(
    bands=['B4', 'B3', 'B2'],
    min=0,
    max=3000
))

In [10]:

import datetime


now = datetime.datetime.now()
time_stamp = now.strftime("%Y_%m_%d_%H_%M_%S")

file_name = 'My_First_Satellite_Timelapse_' + time_stamp
print('Exporting file as:', file_name)

# Create the export task
task = ee.batch.Export.video.toDrive(
    collection=visualized_collection,
    folder='GEE_Exports',
    description=file_name,
    dimensions=720,    
    framesPerSecond=10, 
    region=region
)

task.start()

Exporting file as: My_First_Satellite_Timelapse_2026_01_03_01_26_28


In [ ]:
try:
    while task.active():
        print('Polling for task (id: {task.id}). Current status: {task.status()}')
        import time
        time.sleep(30)
except KeyboardInterrupt:
    print('stopped')

Polling for task (id: AACRYAI7XPYLTI3K53KRGYGS). Current status: RUNNING
Polling for task (id: AACRYAI7XPYLTI3K53KRGYGS). Current status: RUNNING
Polling for task (id: AACRYAI7XPYLTI3K53KRGYGS). Current status: RUNNING


Trying to save locally

In [7]:
work_dir = os.path.join(os.getcwd(), '3 Timelapse')
if not os.path.exists(work_dir):
    os.makedirs(work_dir)


print("Downloading individual frames to the folder")

geemap.download_ee_image_collection(
    visualized_collection,
    out_dir=work_dir,
    region=region, 
    scale=10, #this is because Sentinel-2 has 10m resolution for RGB bands
)



Total number of images: 321



20160118T043122_20160118T043403_T46QBM:   0%|          |0/15 tiles [00:00<?]

20160118T043122_20160118T043403_T45QZG:   0%|          |0/15 tiles [00:00<?]

20170102T043152_20170102T043712_T46QBM:   0%|          |0/15 tiles [00:00<?]

20170102T043152_20170102T043712_T45QZG:   0%|          |0/15 tiles [00:00<?]

...

20251211T043059_20251211T044101_T45QZG:   0%|          |0/15 tiles [00:00<?]

20251216T043211_20251216T043557_T46QBM:   0%|          |0/15 tiles [00:00<?]

20251216T043211_20251216T043557_T45QZG:   0%|          |0/15 tiles [00:00<?]

20251218T043231_20251218T043229_T46QBM:   0%|          |0/15 tiles [00:00<?]

20251218T043231_20251218T043229_T45QZG:   0%|          |0/15 tiles [00:00<?]